# Vision Zero LA — Fabric GeoAnalytics Pipeline

**Purpose:** Ingest LAPD traffic collision data, process with GeoAnalytics and H3 spatial indexing, and produce pre-computed outputs for the CityPulse visualization app.

**Data Source:** [LAPD Traffic Collision Data](https://data.lacity.org/Public-Safety/Traffic-Collision-Data-from-2010-to-Present/d5tf-ez2w) — 700K+ records since 2010, updated weekly.

**Outputs (written to Lakehouse):**
- `crash_h3_heatmap` — H3 hex aggregations with crash counts by severity
- `danger_corridors` — Road segments colored by crash density
- `crash_points` — Individual crashes with severity classification
- `crash_stats` — Summary statistics and top dangerous intersections
- `crash_timeseries` — Monthly crash trends for animated playback

**Schedule:** Run weekly via Fabric Pipeline after LAPD data refresh.

## 1. Setup & Configuration

In [ ]:
# Install H3 if not available
%pip install h3 requests

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import h3
import requests
import json
import math

# Configuration
SODA_BASE = "https://data.lacity.org/resource/d5tf-ez2w.json"
DATE_FROM = "2022-01-01"
H3_RESOLUTION = 9  # ~174m hex edge, city-block level
H3_RESOLUTION_COARSE = 8  # ~460m for zoomed-out view
ROAD_MATCH_DISTANCE_M = 50  # max distance to match crash to road

# Lakehouse paths (adjust to your Lakehouse name)
LAKEHOUSE = "VisionZero"
OUTPUT_PATH = f"Files/citypulse_output"

print(f"Pipeline configured: crashes from {DATE_FROM}, H3 res {H3_RESOLUTION}")

## 2. Ingest — Fetch LAPD Crash Data from SODA API

In [ ]:
def fetch_lapd_crashes(date_from, batch_size=50000):
    """Fetch all LAPD crash records from SODA API with pagination."""
    all_records = []
    offset = 0
    
    while True:
        url = (
            f"{SODA_BASE}"
            f"?$where=date_occ>'{date_from}T00:00:00'"
            f"&$limit={batch_size}"
            f"&$offset={offset}"
            f"&$select=dr_no,date_occ,time_occ,area_name,crm_cd_desc,"
            f"vict_age,vict_sex,vict_descent,premis_desc,location,"
            f"cross_street,lat,lon,mocodes"
            f"&$order=date_occ DESC"
        )
        
        resp = requests.get(url)
        resp.raise_for_status()
        batch = resp.json()
        
        if not batch:
            break
            
        all_records.extend(batch)
        print(f"  Fetched {len(all_records)} records (batch {offset // batch_size + 1})")
        
        if len(batch) < batch_size:
            break
        offset += batch_size
    
    return all_records

print(f"Fetching crashes from {DATE_FROM}...")
raw_records = fetch_lapd_crashes(DATE_FROM)
print(f"Total: {len(raw_records)} records")

In [ ]:
# Convert to Spark DataFrame
schema = StructType([
    StructField("dr_no", StringType()),
    StructField("date_occ", StringType()),
    StructField("time_occ", StringType()),
    StructField("area_name", StringType()),
    StructField("crm_cd_desc", StringType()),
    StructField("vict_age", StringType()),
    StructField("vict_sex", StringType()),
    StructField("vict_descent", StringType()),
    StructField("premis_desc", StringType()),
    StructField("location", StringType()),
    StructField("cross_street", StringType()),
    StructField("lat", StringType()),
    StructField("lon", StringType()),
    StructField("mocodes", StringType()),
])

crashes_raw = spark.createDataFrame(raw_records, schema)
print(f"Raw DataFrame: {crashes_raw.count()} rows")
crashes_raw.printSchema()

## 3. Bronze → Silver: Clean & Enrich

In [ ]:
# Register H3 UDFs
@F.udf(StringType())
def lat_lng_to_h3(lat, lng, res):
    try:
        return h3.latlng_to_cell(float(lat), float(lng), int(res))
    except:
        return None

@F.udf(StringType())
def h3_to_boundary_json(h3_index):
    try:
        boundary = h3.cell_to_boundary(h3_index)
        # Return as JSON array of [lng, lat] pairs (GeoJSON order)
        coords = [[round(lng, 6), round(lat, 6)] for lat, lng in boundary]
        coords.append(coords[0])  # close the ring
        return json.dumps(coords)
    except:
        return None

# Classify crash severity from MO codes
@F.udf(StringType())
def classify_severity(mocodes, crm_cd_desc):
    if mocodes is None:
        mocodes = ""
    codes = set(mocodes.strip().split())
    desc = (crm_cd_desc or "").upper()
    
    if "0449" in codes or "KILLED" in desc:
        return "fatal"
    elif "1920" in codes or "PEDESTRIAN" in desc:
        return "pedestrian"
    elif "1822" in codes or "BICYCLE" in desc or "BIKE" in desc:
        return "bicycle"
    elif "0448" in codes or "FELONY" in desc:
        return "severe"
    else:
        return "other"

print("UDFs registered.")

In [ ]:
# Clean and enrich
crashes = (
    crashes_raw
    # Parse types
    .withColumn("latitude", F.col("lat").cast("double"))
    .withColumn("longitude", F.col("lon").cast("double"))
    .withColumn("crash_date", F.to_date("date_occ"))
    .withColumn("crash_hour", F.substring("time_occ", 1, 2).cast("int"))
    .withColumn("crash_year", F.year("crash_date"))
    .withColumn("crash_month", F.month("crash_date"))
    .withColumn("year_month", F.date_format("crash_date", "yyyy-MM"))
    
    # Filter invalid coordinates
    .filter(F.col("latitude").isNotNull())
    .filter(F.col("longitude").isNotNull())
    .filter(F.col("latitude") != 0)
    .filter(F.col("longitude") != 0)
    .filter(F.col("latitude").between(33.5, 35.0))
    .filter(F.col("longitude").between(-119.0, -117.5))
    
    # Classify severity
    .withColumn("severity", classify_severity(F.col("mocodes"), F.col("crm_cd_desc")))
    
    # H3 indexing at two resolutions
    .withColumn("h3_res9", lat_lng_to_h3(F.col("latitude"), F.col("longitude"), F.lit(9)))
    .withColumn("h3_res8", lat_lng_to_h3(F.col("latitude"), F.col("longitude"), F.lit(8)))
    
    # Intersection label for stats
    .withColumn("intersection", 
        F.when(F.col("cross_street").isNotNull() & (F.col("cross_street") != ""),
               F.concat(F.col("location"), F.lit(" / "), F.col("cross_street")))
        .otherwise(F.col("location"))
    )
    
    .filter(F.col("h3_res9").isNotNull())
    .cache()
)

total = crashes.count()
print(f"Cleaned crashes: {total}")
crashes.groupBy("severity").count().orderBy(F.desc("count")).show()

In [ ]:
# Save to Lakehouse as Delta table (Bronze/Silver)
crashes.write.format("delta").mode("overwrite").saveAsTable(f"{LAKEHOUSE}.silver_crashes")
print("Saved to silver_crashes Delta table.")

## 4. Gold: H3 Hexagonal Heatmap

In [ ]:
def build_h3_heatmap(df, h3_col, resolution_label):
    """Aggregate crashes by H3 cell, compute stats, generate boundary polygons."""
    
    heatmap = (
        df.groupBy(h3_col)
        .agg(
            F.count("*").alias("crash_count"),
            F.sum(F.when(F.col("severity") == "fatal", 1).otherwise(0)).alias("fatal_count"),
            F.sum(F.when(F.col("severity") == "pedestrian", 1).otherwise(0)).alias("pedestrian_count"),
            F.sum(F.when(F.col("severity") == "bicycle", 1).otherwise(0)).alias("bicycle_count"),
            F.sum(F.when(F.col("severity") == "severe", 1).otherwise(0)).alias("severe_count"),
            F.avg("latitude").alias("center_lat"),
            F.avg("longitude").alias("center_lng"),
        )
        .withColumn("boundary", h3_to_boundary_json(F.col(h3_col)))
        .filter(F.col("boundary").isNotNull())
        .withColumnRenamed(h3_col, "h3_index")
        .withColumn("resolution", F.lit(resolution_label))
    )
    
    return heatmap

# Build both resolutions
heatmap_r9 = build_h3_heatmap(crashes, "h3_res9", "9")
heatmap_r8 = build_h3_heatmap(crashes, "h3_res8", "8")
heatmap = heatmap_r9.unionByName(heatmap_r8)

print(f"H3 cells: {heatmap_r9.count()} (res 9) + {heatmap_r8.count()} (res 8)")
heatmap.orderBy(F.desc("crash_count")).show(10)

In [ ]:
heatmap.write.format("delta").mode("overwrite").saveAsTable(f"{LAKEHOUSE}.gold_h3_heatmap")
print("Saved to gold_h3_heatmap.")

## 5. Gold: Danger Corridors (Road-Crash Matching)

In [ ]:
# Load road network (same data the CityPulse app uses)
import os

# Upload la-county.json to Lakehouse Files first, or read from local
road_data_path = f"/lakehouse/default/{OUTPUT_PATH}/la-county.json"

# If running locally for testing, use the repo path instead:
# road_data_path = "../data/la-county.json"

try:
    with open(road_data_path) as f:
        roads = json.load(f)
    print(f"Loaded {len(roads)} road segments")
except FileNotFoundError:
    print(f"Road file not found at {road_data_path}.")
    print("Upload la-county.json to Lakehouse Files/citypulse_output/ first.")
    roads = []

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Approximate distance in meters between two lat/lng points."""
    R = 6371000
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2 + 
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def point_to_segment_dist(px, py, ax, ay, bx, by):
    """Approximate distance from point to line segment in degrees (fast, for filtering)."""
    dx, dy = bx - ax, by - ay
    if dx == 0 and dy == 0:
        return math.sqrt((px - ax)**2 + (py - ay)**2)
    t = max(0, min(1, ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)))
    proj_x, proj_y = ax + t * dx, ay + t * dy
    return math.sqrt((px - proj_x)**2 + (py - proj_y)**2)

# Build spatial grid index of crashes for fast lookup
GRID_SIZE = 0.005  # ~500m grid cells

crash_points = crashes.select("latitude", "longitude", "severity").collect()
print(f"Building spatial index for {len(crash_points)} crashes...")

crash_grid = {}  # (grid_x, grid_y) -> [(lat, lng, severity), ...]
for row in crash_points:
    gx = int(row.longitude / GRID_SIZE)
    gy = int(row.latitude / GRID_SIZE)
    key = (gx, gy)
    if key not in crash_grid:
        crash_grid[key] = []
    crash_grid[key].append((row.latitude, row.longitude, row.severity))

print(f"Grid cells: {len(crash_grid)}")

In [ ]:
# Match crashes to road segments
MATCH_THRESHOLD_DEG = 0.0005  # ~55m in degrees

road_crash_density = []  # [{road_index, crash_count, fatal_count, ...}, ...]

for road_idx, road in enumerate(roads):
    path = road["path"]
    if len(path) < 2:
        continue
    
    # Find all grid cells this road passes through
    road_cells = set()
    for pt in path:
        gx = int(pt[0] / GRID_SIZE)
        gy = int(pt[1] / GRID_SIZE)
        # Check this cell and neighbors
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                road_cells.add((gx + dx, gy + dy))
    
    # Collect candidate crashes from nearby grid cells
    candidates = []
    for cell in road_cells:
        if cell in crash_grid:
            candidates.extend(crash_grid[cell])
    
    if not candidates:
        road_crash_density.append({
            "road_index": road_idx, "crash_count": 0,
            "fatal": 0, "pedestrian": 0, "bicycle": 0, "severe": 0
        })
        continue
    
    # Check each candidate against road segments
    counts = {"total": 0, "fatal": 0, "pedestrian": 0, "bicycle": 0, "severe": 0}
    for clat, clng, sev in candidates:
        min_dist = float("inf")
        for i in range(len(path) - 1):
            d = point_to_segment_dist(clng, clat, path[i][0], path[i][1], path[i+1][0], path[i+1][1])
            min_dist = min(min_dist, d)
            if min_dist < MATCH_THRESHOLD_DEG:
                break
        
        if min_dist < MATCH_THRESHOLD_DEG:
            counts["total"] += 1
            if sev in counts:
                counts[sev] += 1
    
    road_crash_density.append({
        "road_index": road_idx,
        "crash_count": counts["total"],
        "fatal": counts["fatal"],
        "pedestrian": counts["pedestrian"],
        "bicycle": counts["bicycle"],
        "severe": counts["severe"],
    })
    
    if road_idx % 5000 == 0:
        print(f"  Processed {road_idx}/{len(roads)} roads...")

# Compute max for normalization
max_count = max(r["crash_count"] for r in road_crash_density) or 1
print(f"Road-crash matching complete. Max crashes on a single road: {max_count}")
print(f"Roads with crashes: {sum(1 for r in road_crash_density if r['crash_count'] > 0)}")

In [ ]:
# Build danger corridors output: road paths with crash-density colors
def crash_count_to_color(count, max_count):
    """Map crash count to color: green (safe) -> yellow -> red (dangerous)."""
    if count == 0:
        return [0, 100, 60]  # dim green
    
    t = min(1.0, count / max(max_count * 0.3, 1))  # normalize, saturate at 30% of max
    
    if t < 0.5:
        # Green to Yellow
        s = t * 2
        return [int(255 * s), int(200 + 55 * (1 - s)), 0]
    else:
        # Yellow to Red
        s = (t - 0.5) * 2
        return [255, int(200 * (1 - s)), 0]

danger_corridors = []
for rd in road_crash_density:
    idx = rd["road_index"]
    if idx < len(roads):
        color = crash_count_to_color(rd["crash_count"], max_count)
        danger_corridors.append({
            "path": roads[idx]["path"],
            "color": color,
            "crashes": rd["crash_count"],
            "fatal": rd["fatal"],
            "pedestrian": rd["pedestrian"],
            "bicycle": rd["bicycle"],
        })

print(f"Danger corridors: {len(danger_corridors)} roads")
print(f"  With crashes: {sum(1 for d in danger_corridors if d['crashes'] > 0)}")
print(f"  With fatalities: {sum(1 for d in danger_corridors if d['fatal'] > 0)}")

## 6. Gold: Summary Statistics

In [ ]:
# Overall stats
stats = {
    "total_crashes": crashes.count(),
    "fatal": crashes.filter(F.col("severity") == "fatal").count(),
    "severe": crashes.filter(F.col("severity") == "severe").count(),
    "pedestrian": crashes.filter(F.col("severity") == "pedestrian").count(),
    "bicycle": crashes.filter(F.col("severity") == "bicycle").count(),
    "date_from": DATE_FROM,
    "date_to": crashes.agg(F.max("crash_date")).collect()[0][0].isoformat(),
}

# Top 20 most dangerous intersections
top_intersections = (
    crashes
    .filter(F.col("intersection").isNotNull())
    .groupBy("intersection")
    .agg(
        F.count("*").alias("crash_count"),
        F.sum(F.when(F.col("severity") == "fatal", 1).otherwise(0)).alias("fatal"),
        F.avg("latitude").alias("lat"),
        F.avg("longitude").alias("lng"),
    )
    .orderBy(F.desc("crash_count"))
    .limit(20)
    .collect()
)

stats["top_intersections"] = [
    {
        "name": row.intersection,
        "crashes": row.crash_count,
        "fatal": row.fatal,
        "lat": round(row.lat, 5),
        "lng": round(row.lng, 5),
    }
    for row in top_intersections
]

# Crash type breakdown
stats["by_severity"] = {
    row.severity: row["count"]
    for row in crashes.groupBy("severity").count().collect()
}

print(json.dumps(stats, indent=2, default=str))

## 7. Gold: Monthly Time Series

In [ ]:
# Monthly crash trends for animated playback
timeseries = (
    crashes
    .groupBy("year_month")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("severity") == "fatal", 1).otherwise(0)).alias("fatal"),
        F.sum(F.when(F.col("severity") == "pedestrian", 1).otherwise(0)).alias("pedestrian"),
        F.sum(F.when(F.col("severity") == "bicycle", 1).otherwise(0)).alias("bicycle"),
    )
    .orderBy("year_month")
    .collect()
)

timeseries_data = [
    {
        "month": row.year_month,
        "total": row.total,
        "fatal": row.fatal,
        "pedestrian": row.pedestrian,
        "bicycle": row.bicycle,
    }
    for row in timeseries
]

print(f"Time series: {len(timeseries_data)} months")
for row in timeseries_data[-6:]:
    print(f"  {row['month']}: {row['total']} crashes, {row['fatal']} fatal")

## 8. Export: Write JSON Files for CityPulse App

In [ ]:
# Create output directory in Lakehouse
output_base = f"/lakehouse/default/{OUTPUT_PATH}"
os.makedirs(output_base, exist_ok=True)

# 1. H3 Heatmap
heatmap_data = [
    {
        "h3": row.h3_index,
        "boundary": json.loads(row.boundary),
        "count": row.crash_count,
        "fatal": row.fatal_count,
        "ped": row.pedestrian_count,
        "bike": row.bicycle_count,
        "res": int(row.resolution),
        "lat": round(row.center_lat, 5),
        "lng": round(row.center_lng, 5),
    }
    for row in heatmap.collect()
]

with open(f"{output_base}/vz-heatmap.json", "w") as f:
    json.dump(heatmap_data, f)
print(f"Wrote vz-heatmap.json: {len(heatmap_data)} hexagons")

# 2. Danger Corridors (road paths with crash-density colors)
with open(f"{output_base}/vz-corridors.json", "w") as f:
    json.dump(danger_corridors, f)
print(f"Wrote vz-corridors.json: {len(danger_corridors)} roads")

# 3. Crash Points (for individual markers)
crash_point_data = [
    {
        "lat": round(row.latitude, 5),
        "lng": round(row.longitude, 5),
        "date": row.crash_date.isoformat() if row.crash_date else None,
        "hour": row.crash_hour,
        "sev": row.severity,
        "desc": row.crm_cd_desc,
        "loc": row.location,
        "cross": row.cross_street,
        "ym": row.year_month,
    }
    for row in crashes.select(
        "latitude", "longitude", "crash_date", "crash_hour",
        "severity", "crm_cd_desc", "location", "cross_street", "year_month"
    ).collect()
]

with open(f"{output_base}/vz-crashes.json", "w") as f:
    json.dump(crash_point_data, f)
print(f"Wrote vz-crashes.json: {len(crash_point_data)} points")

# 4. Stats + Time Series
stats["timeseries"] = timeseries_data
with open(f"{output_base}/vz-stats.json", "w") as f:
    json.dump(stats, f, default=str, indent=2)
print(f"Wrote vz-stats.json")

print(f"\nAll outputs written to: {output_base}/")

## 9. Verify Outputs

In [ ]:
# Quick verification
for fname in ["vz-heatmap.json", "vz-corridors.json", "vz-crashes.json", "vz-stats.json"]:
    fpath = f"{output_base}/{fname}"
    size = os.path.getsize(fpath)
    print(f"  {fname}: {size / 1024:.1f} KB")

print(f"\n--- Summary ---")
print(f"Crashes analyzed: {stats['total_crashes']}")
print(f"Date range: {stats['date_from']} to {stats['date_to']}")
print(f"Fatal: {stats['fatal']}")
print(f"Pedestrian: {stats['pedestrian']}")
print(f"Bicycle: {stats['bicycle']}")
print(f"\nTop 5 most dangerous intersections:")
for i, inter in enumerate(stats['top_intersections'][:5]):
    print(f"  {i+1}. {inter['name']}: {inter['crashes']} crashes ({inter['fatal']} fatal)")

print(f"\n✓ Pipeline complete. Download JSON files from Lakehouse and add to CityPulse app.")
print(f"  Or serve directly via Fabric SQL endpoint / OneLake API.")

## Next Steps

1. **Schedule this notebook** via Fabric Pipeline to run weekly
2. **Serve the JSON outputs** via:
   - Download from Lakehouse → add to GitHub repo (`data/` folder)
   - Or expose via Fabric SQL Endpoint for live queries
   - Or copy to Azure Blob Storage with public read access
3. **Update CityPulse app** to load `vz-*.json` files and render:
   - `vz-corridors.json` → animated trails with crash-density colors
   - `vz-heatmap.json` → H3 hexagonal overlay
   - `vz-crashes.json` → individual crash markers with popups
   - `vz-stats.json` → stats panel + time series playback